# PV056 — Plant Disease Classification with Triplet Loss

**Course**: PV056 Machine Learning and Data Mining, MUNI 2026  
**Task**: Classify plant diseases (subtask a) and detect unknown diseases (subtask b) using metric learning on PlantVillage.  
**Repo**: https://github.com/MrJoeKr/pv056-project-2026

> **Runtime**: Set to **GPU** (Runtime → Change runtime type → T4 GPU) before running.  
> **Fast mode** (default): ResNet18 + reduced config, full notebook runs in ~10 min on a T4. Set `FAST_MODE = False` in section 4 to use the full ResNet50 config from the report.

---
## Contents
1. [Setup](#setup) — clone repo, install dependencies
2. [Dataset](#dataset) — download PlantVillage from Kaggle
3. [EDA](#eda) — class distribution, outlier detection
4. [Training](#training) — stratified CV with triplet loss (Fast mode toggle)
5. [Evaluation](#evaluation) — confusion matrix, Grad-CAM, UMAP, per-class F1
6. [Unknown Detection](#unknown) — Mahalanobis distance, ROC, UMAP

---
## 1. Setup <a id='setup'></a>

In [ ]:
# Clone the project repository
!git clone https://github.com/MrJoeKr/pv056-project-2026
%cd pv056-project-2026

In [ ]:
# Install PyTorch with CUDA and remaining dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r requirements.txt -q

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

---
## 2. Dataset <a id='dataset'></a>

Download the **PlantVillage** dataset (`emmarex/plantdisease`, 15 classes, ~20,638 images) straight from Kaggle using their public API. No Drive mount needed.

### Step 1 — Get your Kaggle API key
1. Sign in at https://www.kaggle.com.
2. Top-right avatar → **Settings**.
3. Scroll to the **API** section → click **Create New Token**. Your browser downloads `kaggle.json` containing `{"username": "...", "key": "..."}`.
4. Open that file and copy the two values.

### Step 2 — Store the key in Colab Secrets
1. In the Colab left sidebar click the 🔑 **key icon** (*Secrets*).
2. Click **+ Add new secret** twice and create:
   - Name `KAGGLE_USERNAME` → value = your Kaggle username.
   - Name `KAGGLE_KEY` → value = the `key` string from `kaggle.json`.
3. For **both** secrets, flip the **Notebook access** toggle ON (otherwise `userdata.get(...)` will raise).
4. Run the two cells below — `userdata.get()` injects the credentials into the environment and `kaggle datasets download` grabs the archive.

> Accept the dataset's terms once on the Kaggle page (https://www.kaggle.com/datasets/emmarex/plantdisease) before downloading, otherwise the API returns `403`.

In [ ]:
import os
from google.colab import userdata

# Read Kaggle credentials from Colab Secrets (left sidebar, key icon).
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

!pip install -q kaggle

In [ ]:
import os, zipfile

os.makedirs('data', exist_ok=True)

if not os.path.isdir('data/PlantVillage'):
    !kaggle datasets download -d emmarex/plantdisease -p data --quiet
    with zipfile.ZipFile('data/plantdisease.zip') as z:
        z.extractall('data')
    os.remove('data/plantdisease.zip')

classes = sorted(os.listdir('data/PlantVillage'))
print(f'Dataset ready — {len(classes)} classes found')
for c in classes:
    print(' ', c)

---
## 3. Exploratory Data Analysis <a id='eda'></a>

- **R1a**: Class label distribution
- **R1b**: Pixel-level outlier detection (z-score)

In [ ]:
!python scripts/01_eda.py

In [ ]:
from IPython.display import display, Image
import glob

for path in sorted(glob.glob('results/plots/eda/*.png')):
    print(path)
    display(Image(path))

---
## 4. Training — Stratified CV <a id='training'></a>

- **R2a**: HPO was run separately (Optuna, 30 trials); best params baked into `src/config.py`
- **R2b**: Training with early stopping; plots training curves per fold

### Fast mode vs full-quality

The full run (ResNet50, 224×224, 50 epochs, 5 folds) takes ~60–90 min on a T4 — too slow for a Colab demo. The cell below writes a **`config_override.json`** that every script in this project honors, swapping in a fast configuration (ResNet18, 128×128, 10 epochs, 2 folds, mixed precision). Set `FAST_MODE = False` to run the full config used in the report.

> The authoritative numbers in the report were produced locally with ResNet50. This notebook reproduces the full pipeline end-to-end as a runnable demo; see the [GitHub repo](https://github.com/MrJoeKr/pv056-project-2026) for raw results tables and full-quality checkpoints.

In [ ]:
import json, os

FAST_MODE = True  # set to False to use Config() defaults (ResNet50, 224x224, 50 epochs, 5 folds)

override_path = 'results/tables/config_override.json'
os.makedirs(os.path.dirname(override_path), exist_ok=True)

if FAST_MODE:
    overrides = {
        'backbone': 'resnet18',
        'img_size': 128,
        'epochs': 10,
        'patience': 3,
        'n_folds': 2,
        'batch_size': 128,
        'use_amp': True,
    }
    with open(override_path, 'w') as f:
        json.dump(overrides, f, indent=2)
    print('Fast mode active — overrides written:')
    print(json.dumps(overrides, indent=2))
else:
    if os.path.exists(override_path):
        os.remove(override_path)
    print('Full mode — using Config() defaults')

In [ ]:
!python scripts/02_train.py

In [ ]:
for path in sorted(glob.glob('results/plots/training_curves_fold*.png')):
    print(path)
    display(Image(path))

cv_path = 'results/plots/cv_results.png'
if os.path.exists(cv_path):
    print(cv_path)
    display(Image(cv_path))

---
## 5. Evaluation <a id='evaluation'></a>

- **R3a**: Confusion matrix, per-class F1, Grad-CAM explainability, UMAP embedding visualization
- **R3b**: Summary table, statistical context

In [ ]:
!python scripts/04_evaluate.py

In [ ]:
import pandas as pd

# Summary table
summary = pd.read_csv('results/tables/results_summary.csv')
display(summary)

# Plots
for path in ['results/plots/evaluation/confusion_matrix.png',
             'results/plots/evaluation/per_class_f1.png',
             'results/plots/evaluation/umap_embeddings.png',
             'results/plots/evaluation/gradcam_samples.png']:
    if os.path.exists(path):
        print(path)
        display(Image(path))

---
## 6. Unknown Disease Detection <a id='unknown'></a>

Subtask b: `Tomato_Bacterial_spot` is excluded from training and treated as the unknown class.  
Detection uses **Mahalanobis distance** from test embeddings to class prototypes (not softmax).

Results from our run:
- AUROC: **0.9795**, PR-AUC: **0.9627**
- Mann-Whitney U p ≈ 0.00 (unknown distances stochastically larger, highly significant)

In [ ]:
!python scripts/05_unknown.py

In [ ]:
for path in sorted(glob.glob('results/plots/unknown/unknown_*.png')):
    print(path)
    display(Image(path))